# Silver Layer - Hospital Analytics


## Read Bronze Delta Tables

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

patient_bronze = spark.read.format("delta").load(
    "/Volumes/workspace/default/final_assignment/bronze/patients"
)

encounter_bronze = spark.read.format("delta").load(
    "/Volumes/workspace/default/final_assignment/bronze/encounters"
)

claim_bronze = spark.read.format("delta").load(
    "/Volumes/workspace/default/final_assignment/bronze/claims"
)

diagnoses_bronze = spark.read.format("delta").load(
    "/Volumes/workspace/default/final_assignment/bronze/diagnoses"
)

provider_bronze = spark.read.format("delta").load(
    "/Volumes/workspace/default/final_assignment/bronze/providers"
)

In [0]:
patient_silver = (
    patient_bronze
    .withColumn(
        "name",
        concat_ws(
            " ",
            col("first_name"),
            col("last_name")
        )
    )
)

## Create age_group (CASE WHEN)

In [0]:
patient_silver = (
    patient_silver
    .withColumn(
        "age_group", 
        when(col("age") < 18,"Child")
        .when(col("age") < 40, "Young Adult")
        .when(col("age") < 60, "Adult")
        .otherwise("Senior")
    )
)

## Final Patient Columns

In [0]:
patient_silver = patient_silver.select(
    "patient_id",
    "name",
    "age",
    "city",
    "age_group",
    "load_ts"
)

In [0]:
diagnoses_bronze = diagnoses_bronze.withColumnRenamed(
    "load_ts",
    "diagnosis_load_ts"
)

claims_bronze = claim_bronze.withColumnRenamed(
    "load_ts",
    "claim_load_ts"
)

In [0]:
visits_silver = (
    encounter_bronze.join(
        diagnoses_bronze,
        "encounter_id",
        "left"
    )
    .join(
        claims_bronze,
        "encounter_id",
        "left"
    )
)

## Handle Missing Diagnosis

In [0]:
visits_silver = (
    visits_silver
    .withColumn(
        "diagnosis_code",
        coalesce(
            col("diagnosis_code"),
            lit("UNKNOWN")
        )
    )
)

## Rename Columns

In [0]:
visits_silver = visits_silver.select(
    col("encounter_id").alias("visit_id"),
    "patient_id",
    col("provider_id").alias("doctor_id"),
    "visit_date",
    col("diagnosis_code").alias("diagnosis"),
    col("billed_amount").alias("bill_amount"),
    "load_ts"
)

## Remove Duplicate Visits

In [0]:
visits_silver = visits_silver.dropDuplicates(
    ["visit_id"]
)

## Validate Silver Tables

In [0]:
patient_silver.printSchema()

display(patient_silver.limit(10))

print("Patient Count:", patient_silver.count())

root
 |-- patient_id: string (nullable = true)
 |-- name: string (nullable = false)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- age_group: string (nullable = false)
 |-- load_ts: timestamp (nullable = true)



patient_id,name,age,city,age_group,load_ts
PAT000001,Danielle Johnson,85,Bakersfield,Senior,2026-06-23T19:32:36.705Z
PAT000002,Anna Baldwin,15,Bakersfield,Child,2026-06-23T19:32:36.705Z
PAT000003,James Jones,4,Portland,Child,2026-06-23T19:32:36.705Z
PAT000004,Veronica Bowman,48,Seattle,Adult,2026-06-23T19:32:36.705Z
PAT000005,Carl Gentry,83,null,Senior,2026-06-23T19:32:36.705Z
PAT000006,Brenda Hurst,33,Sacramento,Young Adult,2026-06-23T19:32:36.705Z
PAT000007,Kelly Moore,86,Long Beach,Senior,2026-06-23T19:32:36.705Z
PAT000008,Natasha Decker,75,Los Angeles,Senior,2026-06-23T19:32:36.705Z
PAT000009,Rachel Hayes,31,Seattle,Young Adult,2026-06-23T19:32:36.705Z
PAT000010,Amber Cummings,75,San Francisco,Senior,2026-06-23T19:32:36.705Z


Patient Count: 60000


In [0]:
visits_silver.printSchema()

display(visits_silver.limit(10))

print("Visit Count:", visits_silver.count())

root
 |-- visit_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- diagnosis: string (nullable = false)
 |-- bill_amount: double (nullable = true)
 |-- load_ts: timestamp (nullable = true)



visit_id,patient_id,doctor_id,visit_date,diagnosis,bill_amount,load_ts
ENC056041,PAT000003,PRO00055,2025-01-27,S09.90XA,689.65,2026-06-23T19:32:41.529Z
ENC031730,PAT000016,PRO00596,2025-03-18,K58.9,3105.51,2026-06-23T19:32:41.529Z
ENC067209,PAT000017,PRO00075,2025-03-07,E11.9,180.89,2026-06-23T19:32:41.529Z
ENC043617,PAT000025,PRO01153,2025-01-18,N40.0,847.82,2026-06-23T19:32:41.529Z
ENC006314,PAT000035,PRO00142,2025-01-12,E11.9,1758.06,2026-06-23T19:32:41.529Z
ENC024657,PAT000045,PRO00897,2025-01-13,N92.6,964.62,2026-06-23T19:32:41.529Z
ENC050007,PAT000060,PRO00840,2025-03-17,Z00.129,1885.95,2026-06-23T19:32:41.529Z
ENC053951,PAT000067,PRO00056,2025-01-02,R07.9,653.07,2026-06-23T19:32:41.529Z
ENC058611,PAT000068,PRO01453,2025-01-04,G47.00,1303.23,2026-06-23T19:32:41.529Z
ENC002738,PAT000079,PRO00900,2025-02-08,Z34.80,342.37,2026-06-23T19:32:41.529Z


Visit Count: 70000


In [0]:
patient_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/final_assignment/silver/patients")

In [0]:
visits_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/final_assignment/silver/visits")

In [0]:
display(
    spark.read.format("delta")
    .load("/Volumes/workspace/default/final_assignment/silver/patients").limit(10)
)

patient_id,name,age,city,age_group,load_ts
PAT000001,Danielle Johnson,85,Bakersfield,Senior,2026-06-23T19:32:36.705Z
PAT000002,Anna Baldwin,15,Bakersfield,Child,2026-06-23T19:32:36.705Z
PAT000003,James Jones,4,Portland,Child,2026-06-23T19:32:36.705Z
PAT000004,Veronica Bowman,48,Seattle,Adult,2026-06-23T19:32:36.705Z
PAT000005,Carl Gentry,83,null,Senior,2026-06-23T19:32:36.705Z
PAT000006,Brenda Hurst,33,Sacramento,Young Adult,2026-06-23T19:32:36.705Z
PAT000007,Kelly Moore,86,Long Beach,Senior,2026-06-23T19:32:36.705Z
PAT000008,Natasha Decker,75,Los Angeles,Senior,2026-06-23T19:32:36.705Z
PAT000009,Rachel Hayes,31,Seattle,Young Adult,2026-06-23T19:32:36.705Z
PAT000010,Amber Cummings,75,San Francisco,Senior,2026-06-23T19:32:36.705Z


In [0]:
display(
    spark.read.format("delta")
    .load("/Volumes/workspace/default/final_assignment/silver/visits").limit(10)
)

visit_id,patient_id,doctor_id,visit_date,diagnosis,bill_amount,load_ts
ENC056041,PAT000003,PRO00055,2025-01-27,S09.90XA,689.65,2026-06-23T19:32:41.529Z
ENC031730,PAT000016,PRO00596,2025-03-18,K58.9,3105.51,2026-06-23T19:32:41.529Z
ENC067209,PAT000017,PRO00075,2025-03-07,E11.9,180.89,2026-06-23T19:32:41.529Z
ENC043617,PAT000025,PRO01153,2025-01-18,N40.0,847.82,2026-06-23T19:32:41.529Z
ENC006314,PAT000035,PRO00142,2025-01-12,E11.9,1758.06,2026-06-23T19:32:41.529Z
ENC024657,PAT000045,PRO00897,2025-01-13,N92.6,964.62,2026-06-23T19:32:41.529Z
ENC050007,PAT000060,PRO00840,2025-03-17,Z00.129,1885.95,2026-06-23T19:32:41.529Z
ENC053951,PAT000067,PRO00056,2025-01-02,R07.9,653.07,2026-06-23T19:32:41.529Z
ENC058611,PAT000068,PRO01453,2025-01-04,G47.00,1303.23,2026-06-23T19:32:41.529Z
ENC002738,PAT000079,PRO00900,2025-02-08,Z34.80,342.37,2026-06-23T19:32:41.529Z
